# GnitzDB — SQL quickstart: incremental materialized views

**GnitzDB** is a streaming SQL database built on **DBSP**. A `CREATE VIEW`
compiles to a circuit that gnitz maintains **incrementally**: when base data
changes, it updates the view from just the *delta* rather than recomputing it.
Internally every relation is a **Z-set** — each row carries an integer *weight* —
the motto being *"ZSets all the way down."*

Using plain SQL through the Python bindings, this notebook will:

1. Connect to a running `gnitz-server` over its Unix domain socket.
2. Create two tables (`customers`, `orders`).
3. Define a **JOIN** view, then a **GROUP BY** view layered on top of it.
4. Read the maintained results with ordinary `SELECT`.
5. Query with `SELECT ... WHERE ...`.
6. Insert more data and watch a view update incrementally.

> Everything goes through `conn.execute_sql(...)`. gnitz also has a low-level
> Python *circuit-builder* API, which we deliberately don't use here.

## 0. Start a server

gnitz is a client/server engine; the client talks to a `gnitz-server` process
over a **Unix domain socket** (no TCP host/port). From the repository root:

```bash
# Build the server binary and the Python extension (the `gnitz` package).
make server
make pyext

# Launch a server:  gnitz-server <data_dir> <socket_path> [--workers=N]
./gnitz-server /tmp/gnitz-demo-data /tmp/gnitz-demo.sock --workers=4
```

Leave that running. The socket path you pass the server is the one this notebook
connects to (override it with the `GNITZ_SOCK` environment variable).

**Kernel:** the notebook's kernel must be able to `import gnitz` — the simplest
way is to launch Jupyter from the extension's environment:

```bash
cd crates/gnitz-py && uv run jupyter lab
```

In [ ]:
import os

import gnitz

# The server's Unix-socket path — must match the path gnitz-server was started with.
SOCK_PATH = os.environ.get("GNITZ_SOCK", "/tmp/gnitz-demo.sock")

# Tables and views live in a schema; a `public` schema exists out of the box.
SCHEMA = "public"

print("will connect to", SOCK_PATH)

## 1. Connect

`gnitz.connect(socket_path)` returns a `Connection` (also usable as a context
manager: `with gnitz.connect(path) as conn: ...`). We keep a long-lived handle
here and close it at the end.

In [ ]:
conn = gnitz.connect(SOCK_PATH)
print("connected:", conn)

### (optional) Clean slate

So the notebook can be re-run against the same server, drop anything left over
from a previous run.

In [ ]:
# Re-run friendly: drop anything left from a previous run. Views are dropped
# before the tables they read, and the GROUP BY view before the join view it
# builds on. Dropping something absent just raises, which we ignore.
for stmt in (
    "DROP VIEW revenue_by_city",
    "DROP VIEW big_orders",
    "DROP VIEW order_details",
    "DROP TABLE orders",
    "DROP TABLE customers",
):
    try:
        conn.execute_sql(stmt, schema_name=SCHEMA)
    except gnitz.GnitzError:
        pass  # didn't exist yet — fine on a fresh server

print("clean slate ready")

## 2. Create two tables

The column types here are `BIGINT` (64-bit integer) and `TEXT` (string); the
primary key is `id`. `execute_sql` returns one result dict per statement — a
`CREATE TABLE` yields `{'type': 'TableCreated', 'table_id': ...}`.

In [ ]:
conn.execute_sql(
    """
    CREATE TABLE customers (
        id   BIGINT NOT NULL PRIMARY KEY,
        name TEXT   NOT NULL,
        city TEXT   NOT NULL
    )
    """,
    schema_name=SCHEMA,
)
conn.execute_sql(
    """
    CREATE TABLE orders (
        id          BIGINT NOT NULL PRIMARY KEY,
        customer_id BIGINT NOT NULL,
        amount      BIGINT NOT NULL
    )
    """,
    schema_name=SCHEMA,
)
print("tables created")

## 3. Define a view with a JOIN

A view is a standing query that gnitz keeps up to date automatically: as base
data changes it maintains the view incrementally, updating it from each change
rather than recomputing.

Each view is a single relational *shape* — a filter/projection, a single JOIN,
or a GROUP BY. So we start with the join, pairing every order with its customer.
(A join view can't carry its own `WHERE`/`GROUP BY`; those live in a view layered
on top — the next step.)

In [ ]:
conn.execute_sql(
    """
    CREATE VIEW order_details AS
    SELECT customers.name AS name,
           customers.city AS city,
           orders.amount  AS amount
    FROM orders
    JOIN customers ON orders.customer_id = customers.id
    """,
    schema_name=SCHEMA,
)
print("view order_details created")

## 4. Layer a GROUP BY view on top

Because JOIN and GROUP BY are different shapes, *join then aggregate* is
expressed as a `GROUP BY` view that reads **from the join view**. This one rolls
the joined rows up to revenue per city. `SUM` / `COUNT` / `MIN` / `MAX` are all
available; a `GROUP BY` view's SELECT list may contain only the group column(s)
and aggregates.

In [ ]:
conn.execute_sql(
    """
    CREATE VIEW revenue_by_city AS
    SELECT city,
           SUM(amount) AS total,
           COUNT(*)    AS order_count
    FROM order_details
    GROUP BY city
    """,
    schema_name=SCHEMA,
)
print("view revenue_by_city created")

## 5. Insert some data

`execute_sql` runs DML too; an `INSERT` returns
`{'type': 'RowsAffected', 'count': N}`. The views already exist, so each `INSERT`
is folded into them incrementally.

> The order isn't important for *correctness*: a view created over tables that
> already hold rows is backfilled from them and then maintained incrementally.
> We create the views first here simply so the inserts below flow through them.

In [ ]:
conn.execute_sql(
    """
    INSERT INTO customers VALUES
        (1, 'Alice', 'Berlin'),
        (2, 'Bob',   'Hamburg'),
        (3, 'Carol', 'Berlin'),
        (4, 'Dave',  'Hamburg'),
        (5, 'Erin',  'Munich')
    """,
    schema_name=SCHEMA,
)
conn.execute_sql(
    """
    INSERT INTO orders VALUES
        (101, 1, 250),
        (102, 1, 125),
        (103, 2, 400),
        (104, 3,  90),
        (105, 4, 510),
        (106, 5, 300),
        (107, 2, 150)
    """,
    schema_name=SCHEMA,
)
print("rows inserted")

## 6. Read the views

Read a view by **name** with an ordinary `SELECT * FROM <view>` — there's no need
to track view ids. gnitz returns the maintained rows; it also carries one
internal key column (named with a leading `_`), which the small helper below
hides.

With the data above, `revenue_by_city` reads **Berlin** 465 / 3 orders,
**Hamburg** 1060 / 3 orders, **Munich** 300 / 1 order.

In [ ]:
def query(sql):
    # Run a SELECT and return the rows as dicts, hiding gnitz's internal
    # key column (named with a leading underscore).
    result = conn.execute_sql(sql, schema_name=SCHEMA)
    return [
        {k: v for k, v in row._asdict().items() if not k.startswith("_")}
        for row in result[0]["rows"]
    ]


print("order_details (the JOIN view):")
for row in sorted(query("SELECT * FROM order_details"), key=lambda r: r["name"]):
    print("  ", row)

print("\nrevenue_by_city (the GROUP BY view):")
for row in sorted(query("SELECT * FROM revenue_by_city"), key=lambda r: r["city"]):
    print("  ", row)

## 7. `SELECT ... WHERE ...`

A direct `SELECT` supports a **primary-key point lookup** — `WHERE <pk> = <value>`
(or no `WHERE`, a full scan). To filter on a **non-key** column, express the
predicate as a **view** and read it back by name — that's gnitz's
continuous-query model.

In [ ]:
# A direct SELECT is a primary-key point lookup: WHERE <pk> = <value>.
print("customer 1:", query("SELECT * FROM customers WHERE id = 1"))

# To filter on a non-key column, express the predicate as a view and read it by name.
conn.execute_sql(
    "CREATE VIEW big_orders AS SELECT * FROM orders WHERE amount >= 300",
    schema_name=SCHEMA,
)
print("\norders with amount >= 300:")
for row in sorted(query("SELECT * FROM big_orders"), key=lambda r: r["id"]):
    print("  ", row)

## 8. Watch a view update incrementally

Views stay live. Insert one more order and read the same view again: gnitz folds
the single new row into `revenue_by_city` without recomputing it — Munich goes
from 300 / 1 order to **1000 / 2 orders**.

In [ ]:
conn.execute_sql("INSERT INTO orders VALUES (108, 5, 700)", schema_name=SCHEMA)

print("revenue_by_city after the new Munich order:")
for row in sorted(query("SELECT * FROM revenue_by_city"), key=lambda r: r["city"]):
    print("  ", row)

## 9. Close

`Connection` is also a context manager, so `with gnitz.connect(...) as conn:`
would close it for you. Closing is idempotent.

In [ ]:
conn.close()
print("closed")